# 01 — Download YOLO (person detection, CPU)

Downloads **YOLOv11n** (and optionally YOLOv8n) nano weights for **CPU**.

- Saves to `models/yolo/`
- Exports **ONNX** for ONNX Runtime
- Person-only filtering is applied at inference: `classes=[0]` (COCO person)

Run cells top to bottom.


In [3]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current notebook location.")

ROOT = find_project_root()
YOLO_DIR = ROOT / "models" / "yolo"
YOLO_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("YOLO dir    :", YOLO_DIR)


Project root: /Users/macbookpro/Desktop/person-face-events
YOLO dir    : /Users/macbookpro/Desktop/person-face-events/models/yolo


In [1]:
# Run once if packages are missing
%pip install -q ultralytics onnxruntime opencv-python tqdm


Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import shutil
from ultralytics import YOLO

ROOT = Path("..").resolve()
YOLO_DIR = ROOT / "models" / "yolo"
YOLO_DIR.mkdir(parents=True, exist_ok=True)

def download_yolo(model_name: str = "yolo11n.pt", export_onnx: bool = True, imgsz: int = 416):
    """Download nano YOLO into models/yolo and optionally export ONNX."""
    print(f"\n=== {model_name} ===")
    model = YOLO(model_name)  # downloads weights automatically

    dest_pt = YOLO_DIR / model_name

    # Locate downloaded .pt
    pt_file = None
    if getattr(model, "ckpt_path", None):
        pt_file = Path(model.ckpt_path)
    if pt_file is None or not pt_file.exists():
        try:
            from ultralytics.utils import WEIGHTS_DIR
            cand = Path(WEIGHTS_DIR) / model_name
            if cand.exists():
                pt_file = cand
        except Exception:
            pass
    if pt_file is None or not pt_file.exists():
        # ultralytics also drops weights in CWD sometimes
        cand = Path(model_name).resolve()
        if cand.exists():
            pt_file = cand

    if pt_file and pt_file.exists():
        shutil.copy2(pt_file, dest_pt)
    else:
        raise FileNotFoundError(
            f"Could not locate {model_name} after YOLO() download. "
            "Check network / ultralytics version."
        )

    print(f"Saved PT  : {dest_pt} ({dest_pt.stat().st_size / 1e6:.1f} MB)")

    dest_onnx = None
    if export_onnx:
        print("Exporting ONNX for CPU ...")
        local = YOLO(str(dest_pt))
        out = local.export(format="onnx", imgsz=imgsz, simplify=True, dynamic=False)
        out_path = Path(out)
        dest_onnx = YOLO_DIR / f"{Path(model_name).stem}.onnx"
        if out_path.resolve() != dest_onnx.resolve():
            if dest_onnx.exists():
                dest_onnx.unlink()
            shutil.move(str(out_path), str(dest_onnx))
        print(f"Saved ONNX: {dest_onnx} ({dest_onnx.stat().st_size / 1e6:.1f} MB)")

    return dest_pt, dest_onnx

# Primary CPU model (recommended)
pt, onnx = download_yolo("yolo11n.pt", export_onnx=True, imgsz=416)

# Optional fallback:
# download_yolo("yolov8n.pt", export_onnx=True, imgsz=416)

print("\n✓ YOLO download complete")
print("Set config/settings.yaml -> models.yolo_weights to:")
print(" ", pt)



=== yolo11n.pt ===
Saved PT  : /Users/macbookpro/Desktop/person-face-events/models/yolo/yolo11n.pt (5.6 MB)
Exporting ONNX for CPU ...
Ultralytics 8.4.112 🚀 Python-3.12.10 torch-2.2.2 CPU (Intel Core i9-9880H 2.30GHz)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 2.7 GFLOPs

PyTorch: starting from '/Users/macbookpro/Desktop/person-face-events/models/yolo/yolo11n.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 84, 3549) (5.4 MB)

ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.7s, saved as '/Users/macbookpro/Desktop/person-face-events/models/yolo/yolo11n.onnx' (10.2 MB)

Export complete (3.4s)
Results saved to /Users/macbookpro/Desktop/person-face-events/models/yolo/yolo11n.onnx
Predict:         yolo predict task=detect model=/Users/macbookpro/Desktop/person-face-events/models/yolo/yolo11n.onnx imgsz=416 
Validate:        yolo val task=detect model=/Users/macbookpro/Desktop/p

In [1]:
from pathlib import Path
import numpy as np
from ultralytics import YOLO

root = ROOT

pt = root / "models" / "yolo" / "yolo11n.pt"
if not pt.exists():
    pt = root.parent / "models" / "yolo" / "yolo11n.pt"

model = YOLO(str(pt))
dummy = np.zeros((480, 640, 3), dtype=np.uint8)

res = model.predict(
    source=dummy,
    classes=[0],   # person only
    device="cpu",
    imgsz=416,
    verbose=False,
)

boxes = res[0].boxes
n = 0 if boxes is None else len(boxes)

print(f"Person-only predict OK (boxes on blank frame: {n})")
print("COCO class 0 = person; all other classes are ignored")

Person-only predict OK (boxes on blank frame: 0)
COCO class 0 = person; all other classes are ignored


## Next

1. Run `02_download_bytetrack.ipynb`
2. Run `03_download_face_models.ipynb`
3. Run `04_verify_models_cpu.ipynb`
